In [ ]:
import os
try:
    from google.colab import userdata
    _k = userdata.get("OPENAI_API_KEY")
    if _k:
        os.environ["OPENAI_API_KEY"] = _k
except Exception:
    pass
print("OPENAI_API_KEY loaded:", bool(os.environ.get("OPENAI_API_KEY")))


# EpistemicOps: GRPO Training with Unsloth & TRL

Training pipeline for the EpistemicOps SRE environment:
1. Run **baseline episodes** to establish pre-training reward numbers
2. **Restart runtime** to free GPU memory
3. Load Llama 3.1 8B (4-bit) via Unsloth
4. Train with GRPO using the environment's reward function
5. Compare before/after reward curves

> **Runtime:** Select T4 GPU via Runtime > Change runtime type

In [ ]:
# Install dependencies
!pip install -q openai anthropic wandb
!pip install -q "transformers>=4.51.3,<=5.5.0" trl datasets httpx pydantic pyyaml matplotlib numpy
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# Clone repo, force-sync branch, and set environment
import os, subprocess

REPO_DIR = '/content/EpistemicOPS'
TARGET_BRANCH = os.getenv('EPISTEMICOPS_BRANCH', 'master')

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/divyam-r25/EpistemicOPS.git', REPO_DIR], check=True)

subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all', '--prune'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', TARGET_BRANCH], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', TARGET_BRANCH], check=True)

os.chdir(REPO_DIR)
os.environ['EPISTEMICOPS_OFFLINE'] = 'true'

os.environ.setdefault('JUDGE_PROVIDER', 'openai')
os.environ.setdefault('JUDGE_MODEL', 'gpt-4o-mini')
os.environ.setdefault('PRIMARY_AGENT_MODEL', 'gpt-4o-mini')

commit = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD']).decode().strip()
print(f'Working dir: {os.getcwd()}')
print(f'Checked out branch: {TARGET_BRANCH}')
print(f'Commit: {commit}')
print('Model defaults: PRIMARY_AGENT_MODEL=gpt-4o-mini, JUDGE_PROVIDER=openai, JUDGE_MODEL=gpt-4o-mini')
print('Set OPENAI_API_KEY in Colab Secrets or os.environ before LLM-enabled runs.')
print('Note: if you changed code definitions, restart runtime before re-importing modules.')

In [ ]:
# Weights & Biases (optional — structured metrics for judges)
import os

os.environ.setdefault("WANDB_PROJECT", "epistemicops")
os.environ.setdefault("WANDB_RUN_GROUP", "grpo-colab")

try:
    import wandb
    from google.colab import userdata
    _wk = userdata.get("WANDB_API_KEY", None)
    if _wk:
        os.environ["WANDB_API_KEY"] = _wk
    if os.environ.get("WANDB_API_KEY"):
        wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
        print("W&B: logged in. GRPO cell will use report_to='wandb'. Project:", os.environ.get("WANDB_PROJECT"))
    else:
        os.environ["WANDB_DISABLED"] = "1"
        print("W&B: no WANDB_API_KEY in Colab Secrets — GRPO uses report_to='none'. Add the secret for live training dashboards.")
except Exception as _e:
    os.environ["WANDB_DISABLED"] = "1"
    print("W&B: skipped (", _e, ") — GRPO uses report_to='none'.")


---
## Step 1: Baseline Evaluation (Before Training)
Run episodes with the mock agent to measure untrained performance.  
Results are saved to `baseline_results.json` so they survive runtime restart.

In [ ]:
import asyncio, json, sys, random, inspect
sys.path.insert(0, '.')

from run_episode import run_full_episode

# Compatibility guardrail: fail fast if stale imports or old repo version are loaded.
sig = inspect.signature(run_full_episode)
accepted_params = list(sig.parameters.keys())
print('run_full_episode params:', accepted_params)
required_params = {'primary_profile', 'primary_use_llm'}
if not required_params.issubset(set(accepted_params)):
    raise RuntimeError(
        'Stale run_episode.py detected. Restart runtime, rerun sync cell, then rerun this cell. '
        f'Missing params: {sorted(required_params - set(accepted_params))}'
    )

# Keep evaluation settings explicit and reused later for apples-to-apples comparison.
EVAL_SCENARIOS = ['cascading_incident']
EVAL_RUNS_PER_SCENARIO = 10
EVAL_ERAS_PER_RUN = 2
SEED_BASE = 7

baseline_rewards = []
baseline_records = []
for scenario_id in EVAL_SCENARIOS:
    for i in range(EVAL_RUNS_PER_SCENARIO):
        random.seed(SEED_BASE + i * 42)
        episode = await run_full_episode(
            scenario_id=scenario_id,
            num_eras=EVAL_ERAS_PER_RUN,
            primary_profile='baseline',
            primary_use_llm=False,
        )
        avg_r = episode['avg_normalized_reward']
        baseline_rewards.append(avg_r)
        baseline_records.append({
            'scenario_id': scenario_id,
            'run_idx': i + 1,
            'avg_normalized_reward': avg_r,
        })
        print(f'[{scenario_id}] Episode {i+1}: R_normalized = {avg_r:.4f}')

baseline_mean = sum(baseline_rewards) / len(baseline_rewards)
print(f'\nBaseline mean: {baseline_mean:.4f}')

# Save to disk so it survives runtime restart
with open('baseline_results.json', 'w') as f:
    json.dump({
        'baseline_rewards': baseline_rewards,
        'baseline_records': baseline_records,
        'mean': baseline_mean,
        'eval_config': {
            'scenarios': EVAL_SCENARIOS,
            'runs_per_scenario': EVAL_RUNS_PER_SCENARIO,
            'eras_per_run': EVAL_ERAS_PER_RUN,
            'seed_base': SEED_BASE,
        },
    }, f, indent=2)
print('Saved to baseline_results.json')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')
ax.bar(range(1, len(baseline_rewards)+1), baseline_rewards,
       color='#e94560', alpha=0.85, label='Baseline (mock agent)')
ax.axhline(y=np.mean(baseline_rewards), color='#FFD700',
           linestyle='--', linewidth=2, label=f'Mean: {np.mean(baseline_rewards):.3f}')
ax.set_xlabel('Episode', color='white', fontsize=12)
ax.set_ylabel('Normalized Reward', color='white', fontsize=12)
ax.set_title('Baseline Performance (No Training)', color='white', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.tick_params(colors='white')
ax.legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white')
ax.grid(axis='y', alpha=0.15, color='white')
plt.tight_layout()
os.makedirs('plots', exist_ok=True)
plt.savefig('plots/colab_baseline.png', dpi=150, facecolor='#1a1a2e')
plt.show()
print('Baseline plot saved.')

---
## IMPORTANT: Restart Runtime Now

The baseline evaluation imported modules that consume GPU memory.  
**You MUST restart the runtime before loading the model.**

Go to **Runtime > Restart session**, then:

1. Re-run **Install dependencies** and **Clone repo / env** cells (and the **W&B setup** + OpenAI secrets cells if you use them).
2. **Do not** re-run the heavy baseline episode loop unless you want fresh baseline numbers.
3. Continue at **Step 2: Load Model** below.

---
## Step 2: Load Model via Unsloth
Run this cell AFTER restarting the runtime.  
It re-initializes the environment and loads the model on a clean GPU.

In [ ]:
# Cell 5: Re-initialize after restart
import os, sys, json
os.chdir('/content/EpistemicOPS')
sys.path.insert(0, '.')
os.environ['EPISTEMICOPS_OFFLINE'] = 'true'

# Load saved baseline results + evaluation config
with open('baseline_results.json') as f:
    saved = json.load(f)

baseline_rewards = saved['baseline_rewards']
baseline_records = saved.get('baseline_records', [])
eval_config = saved.get('eval_config', {
    'scenarios': ['cascading_incident'],
    'runs_per_scenario': 10,
    'eras_per_run': 2,
    'seed_base': 7,
})

print(f'Loaded baseline: mean = {saved["mean"]:.4f} ({len(baseline_rewards)} episodes)')
print('Evaluation config:', eval_config)

In [ ]:
# Cell 6: Load model (T4-safe settings)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    max_seq_length = 1024,
    load_in_4bit = True,
    fast_inference = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Model loaded. Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

---
## Step 3: Prompt Dataset & Reward Function

In [ ]:
# Cell 7: Build prompt dataset
from training.train_primary import build_prompt_dataset, epistemicops_reward_function

dataset = build_prompt_dataset(num_samples=200)
print(f'Dataset: {len(dataset)} prompts')
print(f'Sample (first 200 chars):\n{dataset[0]["prompt"][:200]}...')

In [ ]:
# Cell 8: Verify reward function
test_actions = [
    '{"action_type": "call_tool", "payload": {"tool": "get_incident_status", "args": {"incident_id": "INC-2041"}}}',
    '{"action_type": "declare_hypothesis", "payload": {"hypothesis": "API drift detected", "confidence": 0.7}}',
    '{"action_type": "write_legacy", "payload": {"content": "SECTION 1: state\nSECTION 2: trust\nSECTION 3: drift\nSECTION 4: decisions\nSECTION 5: issues\nSECTION 6: actions"}}',
    'invalid json garbage',
    '{"action_type": "hallucinated_tool", "payload": {}}',
]
rewards = epistemicops_reward_function(test_actions)
for action, r in zip(test_actions, rewards):
    print(f'  R={r:.2f}  {action[:70]}')
print(f'\nReward range: [{min(rewards):.2f}, {max(rewards):.2f}]')

---
## Step 4: GRPO Training

In [ ]:
# Cell 9: Configure and run GRPO training
import os
from trl import GRPOTrainer, GRPOConfig

_report_to = 'none' if os.environ.get('WANDB_DISABLED') == '1' else 'wandb'
print('GRPO report_to:', _report_to)

# Recent TRL: GRPOConfig has no max_prompt_length; prompts are bounded in build_prompt_dataset (legacy_doc[:800]) + tokenizer.
training_args = GRPOConfig(
    output_dir = './checkpoints/primary_agent',
    num_train_epochs = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    learning_rate = 2e-5,
    beta = 0.1,
    temperature = 0.8,
    num_generations = 2,
    max_completion_length = 256,
    logging_steps = 1,
    save_steps = 50,
    bf16 = False,
    fp16 = True,
    report_to = _report_to,
)

trainer = GRPOTrainer(
    model = model,
    reward_funcs = epistemicops_reward_function,
    args = training_args,
    train_dataset = dataset,
    processing_class = tokenizer,
)

print('Starting GRPO training...')
train_result = trainer.train()
print(f'Training complete. Loss: {train_result.training_loss:.4f}')

GRPO report_to: none
Starting GRPO training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
Passing `generation_config` together with generation-related arguments=({'disable_compile', 'pad_token_id', 'cache_implementation'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: Futur

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / epistemicops_reward_function / mean,rewards / epistemicops_reward_function / std
1,0.000000,0.200000,0.282843,203.250000,113.000000,256.000000,0.500000,150.500000,113.000000,188.000000,0.000000,0.200000,0.400000


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / epistemicops_reward_function / mean,rewards / epistemicops_reward_function / std
1,0.000000,0.200000,0.282843,203.250000,113.000000,256.000000,0.500000,150.500000,113.000000,188.000000,0.000000,0.200000,0.400000
2,-0.000000,0.000000,0.000000,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
# Cell 10: Save trained model
model.save_pretrained('./checkpoints/primary_agent_final')
tokenizer.save_pretrained('./checkpoints/primary_agent_final')
print('Model checkpoint saved to ./checkpoints/primary_agent_final')

---
## Step 5: Results & Comparison

In [ ]:
# Cell 11: Training loss curve
import matplotlib.pyplot as plt
import numpy as np

logs = trainer.state.log_history
train_steps = [l['step'] for l in logs if 'loss' in l]
train_losses = [l['loss'] for l in logs if 'loss' in l]

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')
ax.plot(train_steps, train_losses, color='#00d4ff', linewidth=2, marker='o', markersize=4)
ax.set_xlabel('Training Step', color='white', fontsize=12)
ax.set_ylabel('Loss', color='white', fontsize=12)
ax.set_title('GRPO Training Loss Curve', color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='white')
ax.grid(alpha=0.15, color='white')
plt.tight_layout()
plt.savefig('plots/training_loss.png', dpi=150, facecolor='#1a1a2e')
plt.show()

In [ ]:
# Cell 12: Post-training evaluation (episode-level, checkpoint-vs-baseline)
import json
import os
import subprocess
from pathlib import Path

proof_cmd = [
    'python',
    'eval/proof_of_learning.py',
    '--scenarios', ','.join(eval_config['scenarios']),
    '--runs-per-scenario', str(eval_config['runs_per_scenario']),
    '--eras-per-run', str(eval_config['eras_per_run']),
    '--baseline-profile', 'baseline',
    '--trained-agent-source', 'checkpoint',
    '--trained-checkpoint-path', './checkpoints/primary_agent_final',
]
if os.environ.get('WANDB_API_KEY') and os.environ.get('WANDB_DISABLED') != '1':
    proof_cmd.append('--wandb')
    print('Proof will also log eval metrics to W&B (--wandb).')

print('Running proof command:')
print(' '.join(proof_cmd))
subprocess.run(proof_cmd, check=True)

proof_path = Path('eval_results/proof_of_learning.json')
with proof_path.open() as f:
    proof = json.load(f)

summary = proof['summary']
baseline_mean = summary['baseline']['avg_reward']
trained_mean = summary['trained']['avg_reward']
print('\nLoaded proof summary:')
print(json.dumps(summary, indent=2))
print(f'\nBaseline mean reward:  {baseline_mean:.4f}')
print(f'Trained mean reward:   {trained_mean:.4f}')
print(f'Improvement:           {trained_mean - baseline_mean:+.4f}')

In [ ]:
# Cell 13: Final comparison chart (from proof_of_learning.json)
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1a1a2e')

# Left: core metric comparison
ax1.set_facecolor('#16213e')
metric_keys = ['avg_reward', 'avg_criteria_completion', 'drift_detection_rate', 'incident_resolution_rate', 'legacy_doc_rate']
metric_labels = ['Avg Reward', 'Criteria', 'Drift Detect', 'Incident Resolve', 'Legacy Doc']
baseline_vals = [summary['baseline'][k] for k in metric_keys]
trained_vals = [summary['trained'][k] for k in metric_keys]

x = np.arange(len(metric_labels))
w = 0.35
ax1.bar(x - w/2, baseline_vals, w, label='Baseline', color='#e94560', alpha=0.85)
ax1.bar(x + w/2, trained_vals, w, label='Post-GRPO', color='#00d4ff', alpha=0.9)
ax1.set_xticks(x)
ax1.set_xticklabels(metric_labels, rotation=15)
ax1.set_ylim(0, 1.05)
ax1.set_title('Episode-level Metric Comparison', color='white', fontsize=13, fontweight='bold')
ax1.tick_params(colors='white')
ax1.legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white')
ax1.grid(axis='y', alpha=0.15, color='white')

# Right: mean reward summary
ax2.set_facecolor('#16213e')
means = [baseline_mean, trained_mean]
bars = ax2.bar(['Baseline', 'Post-GRPO'], means, color=['#e94560', '#00d4ff'], alpha=0.9)
for bar, m in zip(bars, means):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{m:.3f}', ha='center', color='white', fontsize=14, fontweight='bold')
ax2.set_ylabel('Avg Normalized Reward', color='white', fontsize=12)
ax2.set_title('Mean Reward (Same Env, Same Tasks)', color='white', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 1.0)
ax2.tick_params(colors='white')
ax2.grid(axis='y', alpha=0.15, color='white')

fig.suptitle('EpistemicOps: Baseline vs GRPO Checkpoint (Episode-level)',
             color='white', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/before_after_comparison.png', dpi=150, facecolor='#1a1a2e')
plt.show()
print('Comparison plot saved.')

In [ ]:
# Cell 14: Save run metadata + print trajectory evidence
import json
import platform
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import numpy
import pydantic
import yaml

metadata = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'python_version': platform.python_version(),
    'platform': platform.platform(),
    'offline_mode': os.getenv('EPISTEMICOPS_OFFLINE', ''),
    'eval_config': eval_config,
    'proof_config': proof.get('config', {}),
    'checkpoint_path': './checkpoints/primary_agent_final',
    'summary': summary,
    'deltas': proof.get('deltas', {}),
    'package_versions': {
        'matplotlib': matplotlib.__version__,
        'numpy': numpy.__version__,
        'pydantic': pydantic.__version__,
        'pyyaml': yaml.__version__,
    },
}

# Optional package versions if available in the runtime.
for pkg_name in ['trl', 'transformers', 'datasets', 'torch']:
    try:
        module = __import__(pkg_name)
        metadata['package_versions'][pkg_name] = getattr(module, '__version__', 'unknown')
    except Exception:
        metadata['package_versions'][pkg_name] = 'not_installed'

try:
    metadata['git_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
except Exception:
    metadata['git_commit'] = 'unknown'

try:
    metadata['gpu_name'] = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']).decode().strip()
except Exception:
    metadata['gpu_name'] = 'unknown'

meta_path = Path('eval_results/proof_run_metadata.json')
meta_path.parent.mkdir(parents=True, exist_ok=True)
meta_path.write_text(json.dumps(metadata, indent=2))
print(f'Saved run metadata to {meta_path}')

behavior_path = Path('eval_results/proof_behavior_examples.md')
if behavior_path.exists():
    behavior_text = behavior_path.read_text()
    print('\n=== Behavior Evidence (compact) ===')
    for line in behavior_text.splitlines()[:26]:
        print(line)
else:
    print('Behavior evidence file missing:', behavior_path)


---
## Summary (Judge-facing)

This notebook now reports **episode-level baseline vs trained results on the same environment/tasks** using:
`eval/proof_of_learning.py --trained-agent-source checkpoint --trained-checkpoint-path ./checkpoints/primary_agent_final`

Artifacts produced:
- `eval_results/proof_of_learning.json`
- `eval_results/proof_run_metadata.json`
- `eval_results/proof_behavior_examples.md`
- `plots/proof_reward_curve.png`
- `plots/proof_before_vs_after.png`
- `plots/before_after_comparison.png`

Why this is harder to game:
- Results combine multiple independent reward dimensions (task completion, calibration, teacher delta, legacy utility, leakage, anti-hack).
- Improvement claims are based on **full environment rollouts**, not just single action scoring.
- Behavior excerpts (before/after) are included to validate that reward gains correspond to better drift adaptation.

Full environment details: [HuggingFace Space](https://huggingface.co/spaces/Divyam-r25/EpistemicOps) | [GitHub](https://github.com/divyam-r25/EpistemicOPS)

### Optional: record checkpoint-backed episodes (variance)

After `primary_agent_final` exists and `torch` + `transformers` work in this runtime:

```bash
python training/record_checkpoint_episodes.py --checkpoint ./checkpoints/primary_agent_final --runs 3 --eras 3
```

Writes `episodes/ckpt_<scenario>_runN.json` with different seeds so trajectories are not byte-identical to profile-only runs.
